In [54]:
import pandas as pd
from sklearn.model_selection import train_test_split 

from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight

import numpy as np
import tensorflow as tf



In [55]:
df =pd.read_csv("cremad_features4.csv")

df

,0,1,2,3,4,5,6,7,8,9,...,178,179,180,181,182,183,184,185,label,file_path
0,-306.027405,92.670242,8.491313,23.965405,7.477992,-5.759457,-11.883088,-9.676737,-3.996747,-13.352565,...,0.003217,0.002537,0.101868,1584.993071,143.701602,56.002337,0.041986,0.048556,angry,../AudioWAV/1001_DFA_ANG_XX.wav
1,-346.399628,95.839127,10.516282,31.619215,15.872088,-6.845448,-6.629935,-4.978728,-5.310655,-10.283518,...,0.001274,0.001167,0.093061,1531.650486,140.233641,73.704833,0.015996,0.015574,disgust,../AudioWAV/1001_DFA_DIS_XX.wav
2,-321.420258,94.760918,8.155398,23.323244,11.719157,-7.116333,-8.534803,-4.996965,-4.994401,-13.706510,...,0.010257,0.008362,0.084286,1489.088839,173.526648,58.419397,0.045776,0.061398,fear,../AudioWAV/1001_DFA_FEA_XX.wav
3,-303.303772,92.528893,4.231231,27.970137,10.869824,-11.878345,-10.095113,-7.149731,-7.651760,-17.085903,...,0.003002,0.003396,0.084878,1555.376035,144.048194,67.815715,0.042300,0.048085,happy,../AudioWAV/1001_DFA_HAP_XX.wav
4,-335.495972,100.393318,9.384934,30.160906,11.466775,-3.333670,-8.350987,-9.757346,-6.079329,-12.109532,...,0.000778,0.000657,0.082031,1495.394998,131.235987,60.392878,0.020450,0.022513,neutral,../AudioWAV/1001_DFA_NEU_XX.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7437,-413.367371,96.386276,46.369545,14.591497,20.041862,10.173372,0.748282,-0.852761,4.764163,-0.944673,...,0.001509,0.001905,0.118683,1757.254778,106.960006,61.203559,0.008719,0.006044,disgust,../AudioWAV/1091_WSI_DIS_XX.wav
7438,-426.014099,91.230385,49.762840,20.573893,22.044111,11.756289,-1.450961,2.497097,5.650768,-3.032017,...,0.004655,0.007032,0.096364,1678.540253,121.324500,63.738550,0.008474,0.006287,fear,../AudioWAV/1091_WSI_FEA_XX.wav
7439,-370.487915,90.638107,38.969704,19.762012,14.836708,0.329151,-1.175138,-3.071633,5.731899,-3.763147,...,0.006234,0.006943,0.138205,1851.247161,138.601663,66.646088,0.015657,0.011523,happy,../AudioWAV/1091_WSI_HAP_XX.wav
7440,-393.181274,94.353287,45.251869,12.303626,12.717791,8.813284,0.926235,-3.176066,6.253240,-2.163018,...,0.001125,0.001000,0.113154,1788.313113,137.696471,67.771968,0.011585,0.011870,neutral,../AudioWAV/1091_WSI_NEU_XX.wav


In [56]:
df["label"] = df["label"].map({
    "angry": 0,
    "disgust": 1,
    "fear": 2,
    "sad": 3,
    "neutral": 4,
    "happy": 5,
})


In [57]:
x =df.drop(["label" ,"file_path"] ,axis =1)
y =df["label"]
x_train , x_test ,y_train ,y_test =train_test_split(x ,y ,test_size=0.2 ,random_state=42 ,stratify=y)

In [58]:
scaler_x = StandardScaler()
x_train = scaler_x.fit_transform(x_train)
x_test = scaler_x.transform(x_test)

In [59]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.99)
x_train = pca.fit_transform(x_train)
x_test  = pca.transform(x_test)


In [60]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(
        64, activation="relu",
        kernel_regularizer=tf.keras.regularizers.L2(0.0005),
        input_shape=(x_train.shape[1],)
    ),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(
        32, activation="relu",
        kernel_regularizer=tf.keras.regularizers.L2(0.0001)
    ),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(
        16, activation="relu"
    ),

    tf.keras.layers.Dense(6, activation="softmax")
])



model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00052),loss="sparse_categorical_crossentropy" ,metrics=["accuracy"])

/home/malak/.local/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [61]:
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))


In [62]:
early_stop =tf.keras.callbacks.EarlyStopping(
    monitor ="val_loss",
    patience =10,
    restore_best_weights =True  
)

In [63]:
history =model.fit(x_train ,y_train ,epochs=100 ,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights, 
    callbacks=[early_stop],
    verbose=2)

Epoch 1/100
149/149 - 2s - 16ms/step - accuracy: 0.2289 - loss: 1.9286 - val_accuracy: 0.3123 - val_loss: 1.6842
Epoch 2/100
149/149 - 1s - 4ms/step - accuracy: 0.3213 - loss: 1.6754 - val_accuracy: 0.3652 - val_loss: 1.5921
Epoch 3/100
149/149 - 1s - 4ms/step - accuracy: 0.3610 - loss: 1.5828 - val_accuracy: 0.3879 - val_loss: 1.5275
Epoch 4/100
149/149 - 1s - 4ms/step - accuracy: 0.3858 - loss: 1.5310 - val_accuracy: 0.4089 - val_loss: 1.4850
Epoch 5/100
149/149 - 1s - 4ms/step - accuracy: 0.3990 - loss: 1.4786 - val_accuracy: 0.4374 - val_loss: 1.4528
Epoch 6/100
149/149 - 1s - 4ms/step - accuracy: 0.4229 - loss: 1.4419 - val_accuracy: 0.4442 - val_loss: 1.4317
Epoch 7/100
149/149 - 1s - 5ms/step - accuracy: 0.4410 - loss: 1.4295 - val_accuracy: 0.4551 - val_loss: 1.4090
Epoch 8/100
149/149 - 1s - 5ms/step - accuracy: 0.4536 - loss: 1.3949 - val_accuracy: 0.4610 - val_loss: 1.3918
Epoch 9/100
149/149 - 1s - 7ms/step - accuracy: 0.4702 - loss: 1.3745 - val_accuracy: 0.4685 - val_loss

In [64]:
val_loss, val_acc = model.evaluate(x_train, y_train)
print("Train Accuracy:", val_acc)
print("Train Loss:", val_loss)


187/187 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6657 - loss: 0.9396
Train Accuracy: 0.6657147407531738
Train Loss: 0.9395874738693237


In [65]:
val_loss, val_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", val_acc)
print("Test Loss:", val_loss)


47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5359 - loss: 1.2686
Test Accuracy: 0.5359301567077637
Test Loss: 1.2686063051223755


In [66]:
from sklearn.metrics import f1_score, balanced_accuracy_score

y_pred = model.predict(x_test).argmax(axis=1)

print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))


47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Balanced Accuracy: 0.5381302883550781
Macro F1: 0.5291031473704062
